# Глава 6 — Planning and Reflection

Агент теперь сам продолжает работу после результата инструмента. В одном `run()` он повторяет цикл **модель → действие → observation**, пока не получит окончательный ответ или не исчерпает `max_steps`.

Два варианта: `ReAct` разбирает текстовые `THOUGHT` / `ACTION`, а `NativeReAct` использует структурированные вызовы инструментов. План здесь — следующий шаг, который модель выбирает с учётом полученных результатов; отдельного списка задач в коде нет.

Запустите Jupyter из корня проекта. Примеры используют установленную `gemma4:e4b` и ничего не скачивают. В книге текстовый вариант показан на `gemma3:12b`; здесь у Gemma 4 отключён native reasoning для текстового протокола.

In [ ]:
import math
import socket
from unittest.mock import Mock

from agent import TinyAgent
from llm import LLM, Response
from memory import Memory
from planning import ReAct, NativeReAct
from toolbox import add, multiply, subtract
from tools import Tools, NativeTools
from illustrated_agents.utils import TrajectoryViewer

socket.setdefaulttimeout(120)
MODEL = "gemma4:e4b"
BASE_URL = "http://localhost:11434/v1"
TASK = (
    "Use the available tools to calculate (4.6 + 6.685) * 4 - 3.14. "
    "Perform each arithmetic operation with its tool and give the final answer."
)

def arithmetic_tools(registry_type):
    tools = registry_type()
    for function in (add, multiply, subtract):
        tools.add_tool(function.__name__, function,
                       f"{function.__name__}(a: str, b: str): {function.__doc__}")
    return tools

def check_arithmetic_run(agent, answer):
    steps = agent.trajectory.runs[-1]["steps"]
    actions = [step for step in steps if step.observation is not None]
    assert [step.action["tool"] for step in actions] == ["add", "multiply", "subtract"]
    for step, expected in zip(actions, (11.285, 45.14, 42.0)):
        actual = float(step.observation.removeprefix("OBSERVATION: "))
        assert math.isclose(actual, expected, rel_tol=0, abs_tol=1e-9)
    assert len(steps) == 4 and steps[-1].answer == answer
    assert steps[-1].observation is None and answer and "42" in answer
    print("Проверено: 3 реальных вызова инструментов + финальный ответ.")

## Текстовый ReAct

`ReAct.prompt` задаёт протокол. Модель предлагает одно действие; Python выполняет его и добавляет реальный результат в память. Только после этого модель получает следующий запрос. Текстовый `THOUGHT` — сгенерированное объяснение шага, а не доказательство того, как модель вычисляла ответ внутри.

Завершение обозначается `final_answer`. Это служебный маркер, его не нужно регистрировать как Python-функцию.

In [ ]:
planner = ReAct(max_steps=6)
print(planner.prompt)
agent = TinyAgent(
    llm=LLM(MODEL, base_url=BASE_URL, think=False, temperature=0),
    memory=Memory(),
    tools=arithmetic_tools(Tools),
    planner=planner,
)
answer = agent.run(TASK)
print(answer)
check_arithmetic_run(agent, answer)

## Траектория и память

Ожидаемые observations: `11.285`, `45.14`, `42.0`; затем отдельный шаг с ответом. Проверка смотрит на выполненные действия, а не только на число в финальном тексте.

In [ ]:
print(agent.trajectory.runs)
TrajectoryViewer(agent.trajectory)

In [ ]:
for message in agent.memory.get_messages():
    if message["role"] != "system":
        print(message)

## Native ReAct

Регистрируем те же функции в `NativeTools`, передаём их схемы модели и используем `NativeReAct`. У этого planner нет текстового промпта и разбора `ACTION`. Цикл завершает ответ без `tool_call`; reasoning сохраняется, если сервер его вернул.

`think=True` позволяет backend использовать native reasoning. В памяти сохраняются исходные вызовы и ответы с соответствующими `tool_call_id`.

In [ ]:
native_agent = TinyAgent(
    llm=LLM(MODEL, base_url=BASE_URL, think=True, temperature=0),
    memory=Memory(),
    tools=arithmetic_tools(NativeTools),
    planner=NativeReAct(max_steps=6),
)
native_answer = native_agent.run(TASK)
print(native_answer)
check_arithmetic_run(native_agent, native_answer)

In [ ]:
print(native_agent.trajectory.runs)
TrajectoryViewer(native_agent.trajectory)

## Зачем нужен `max_steps`

Каждый шаг — один запрос генерации и не более одного вызова инструмента. Финальный ответ тоже занимает шаг: для трёх инструментов и ответа нужны минимум четыре шага.

Следующий пример **без модели** имитирует повторяющийся вызов. Он детерминированно проверяет остановку после двух шагов. Результат последнего инструмента остаётся в памяти и траектории, но не выдаётся за окончательный ответ.

In [ ]:
repeating_llm = Mock(spec=LLM)
repeating_llm.generate.return_value = Response(
    'THOUGHT: Add the numbers.\nACTION: {"tool":"add","kwargs":{"a":"1","b":"2"}}'
)
limited_agent = TinyAgent(repeating_llm, tools=arithmetic_tools(Tools), planner=ReAct(max_steps=2))
print(limited_agent.run("Keep adding"))
assert repeating_llm.generate.call_count == 2
assert len(limited_agent.trajectory.runs[0]["steps"]) == 2
assert all(step.answer is None for step in limited_agent.trajectory.runs[0]["steps"])
print(limited_agent.trajectory.runs)

## Память, ошибки и границы реализации

- Без `planner` сохраняется режим главы 5: один запрос генерации, ответ или observation.
- `TrimmingMemory` сохраняет весь текущий ход, включая все действия и observations.
- При planner `SummarizationMemory` сжимает историю только после окончательного ответа. При исчерпании лимита история остаётся. Вызов для сводки — дополнительный запрос, он не считается шагом planner.
- `RAGMemory` ищет документы для исходного пользовательского запроса, а не для каждого observation.
- Ошибки функций и отказ пользователя передаются модели как observations; каждый следующий вызов снова проходит проверку подтверждения. Число попыток ограничено `max_steps`.
- Ошибочный протокол, некорректный JSON, несколько действий или придуманный раздел `OBSERVATION` в текстовом ReAct приводят к исключению до исполнения. Сетевые ошибки также передаются вызывающему коду; автоматических повторов нет.
- `max_steps` ограничивает число генераций, а не время работы инструментов или токены. Здесь зарегистрирована только арифметика.

## Что в главе остаётся теорией

**Self-Refine** использует обратную связь для улучшения ответа, **Reflexion** сохраняет выводы из прошлых попыток, а reinforcement learning меняет поведение через обучение. Здесь реализован кодовый пример главы — ReAct и NativeReAct. Отдельного критика, Reflexion-памяти и обучения модели нет.